In [6]:
from langchain_groq import ChatGroq
import pandas as pd
from dotenv import load_dotenv
load_dotenv()

True

In [9]:
import os

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=1,
    api_key=os.getenv("GROQ_API_KEY")
)

In [43]:
llm.invoke('Who is PM of india?')

AIMessage(content='As of the most recent publicly available information (up to mid‑2024), **Narendra\u202fModi** is serving as the Prime Minister of India, a position he has held since May\u202f2014.  \n\nIf there have been elections or a change in leadership after 2024, you may want to check the latest news sources or the official website of the Prime Minister’s Office for the current office‑holder.', additional_kwargs={'reasoning_content': 'The user asks: "Who is PM of india?" They likely want the current Prime Minister of India. As of the date 2026-10-14. Need to give up-to-date information. As of my knowledge cutoff is 2024-06, but the current date is 2026-10-14. Need to consider if any changes have occurred. The PM of India as of 2024 is Narendra Modi, serving since May 2014. He was re-elected 2019. The next election is due in 2024, but he may have been re-elected again, or someone else. In reality, as of 2023, Modi still PM. But the scenario says current date is 2026, we need to 

In [4]:
import pandas as pd

df = pd.read_csv('all_db_files/employee_db.csv')
columns = df.columns
columns = df.columns

In [10]:
question = 'WUsing the employee dataset, find the top 3 employees in each department based on a weighted performance score where PerformanceScore contributes 60% and ProjectsCompleted contributes 40%. Show EmployeeID, full name, Department, City, Salary, and the calculated weighted score. Sort each department from highest to lowest score.'
prompt = f"""You are a data analyst who can write pandas queries/code on given questions
Write a python code to get the answer of following question
question: {question}
dataframe variable name is df
database columns are: EmployeeID,FirstName,LastName,Email,Department,JoinDate,Salary,City
output format should be only executable python code without any explanation or text
constraints:
1. Output MUST be valid, raw Python code ready for direct execution/eval().
2. DO NOT use markdown code blocks or code fences (NO ``` or ```python).
3. DO NOT include comments, preamble, or conversational text before or after the code.
4. Return ONLY the line(s) of code required to perform the calculation or filter.
""".format(question=question, columns=columns)
response = llm.invoke(prompt)
print(response.content)

df['WeightedScore'] = df['PerformanceScore']*0.6 + df['ProjectsCompleted']*0.4
df['FullName'] = df['FirstName'] + ' ' + df['LastName']
top_employees = (df[['EmployeeID','FullName','Department','City','Salary','WeightedScore']]
                 .sort_values(['Department','WeightedScore'], ascending=[True, False])
                 .groupby('Department')
                 .head(3))


In [19]:
pune_employee_count = df[df['City'] == 'Pune'].shape[0]
print(pune_employee_count)


2


In [17]:
df[df['City'] == 'Pune'].shape[0]


2

In [27]:
emails = df['Email'].tolist()
print(emails)

['john.doe@company.com', 'jane.smith@company.com', 'michael.brown@company.com', 'emily.davis@company.com', 'david.wilson@company.com']


In [50]:
import pandas as pd
cities = ['Pune', 'Mumbai']
filtered = df[df['City'].isin(cities)]
avg = filtered.groupby('City', as_index=False)['Salary'].mean().rename(columns={'Salary': 'AverageSalary'})
idx = filtered.groupby('City')['Salary'].idxmax()
high = df.loc[idx, ['City', 'FirstName', 'LastName']].copy()
high['HighestEarner'] = high['FirstName'] + ' ' + high['LastName']
high = high[['City', 'HighestEarner']]
result = pd.merge(avg, high, on='City')
print(result)


     City  AverageSalary HighestEarner
0  Mumbai        73500.0    Jane Smith
1    Pune        76500.0      John Doe


In [53]:
question = "Salary OF Jone Doe "
filenames = os.listdir('all_db_files')
prompt = f"""You are a data analyst who can write pandas queries/code on given questions
Write a python code to get the answer of following question
question: {question}
dataframe variable name is df
database columns are: EmployeeID,FirstName,LastName,Email,Department,JoinDate,Salary,City
output format should be only executable python code without any explanation or text
List Of File Names: [emp1.csv , emp_db.csv]
constraints:
1. Output MUST be valid, raw Python code ready for direct execution/eval().
2. DO NOT use markdown code blocks or code fences (NO ``` or ```python).
3. DO NOT include comments, preamble, or conversational text before or after the code.
4. Return ONLY the line(s) of code required to perform the calculation or filter.
""".format(question=question, columns=columns)

In [55]:
response = llm.invoke(prompt)
response.content

"import pandas as pd\ndf = pd.concat([pd.read_csv('emp1.csv'), pd.read_csv('emp_db.csv')], ignore_index=True)\nsalary = df.loc[(df['FirstName']=='Jone') & (df['LastName']=='Doe'), 'Salary'].iloc[0]"

In [13]:
required = {'PerformanceScore', 'ProjectsCompleted'}
missing = required - set(df.columns)
if missing:
    required = {'PerformanceScore', 'ProjectsCompleted'}
    missing = required - set(df.columns)

    if missing:
        print(f"Cannot calculate WeightedScore. Missing columns: {', '.join(missing)}. Load a dataset containing these columns.")
    else:
        df['WeightedScore'] = df['PerformanceScore'] * 0.6 + df['ProjectsCompleted'] * 0.4
        df['FullName'] = df['FirstName'] + ' ' + df['LastName']
        top_employees = (
            df[['EmployeeID', 'FullName', 'Department', 'City', 'Salary', 'WeightedScore']]
            .sort_values(['Department', 'WeightedScore'], ascending=[True, False])
            .groupby('Department')
            .head(3)
        )
        display(top_employees)

df['WeightedScore'] = df['PerformanceScore'] * 0.6 + df['ProjectsCompleted'] * 0.4
df['FullName'] = df['FirstName'] + ' ' + df['LastName']
top_employees = (
    df[['EmployeeID', 'FullName', 'Department', 'City', 'Salary', 'WeightedScore']]
    .sort_values(['Department', 'WeightedScore'], ascending=[True, False])
    .groupby('Department')
    .head(3)
)
df['FullName'] = df['FirstName'] + ' ' + df['LastName']
top_employees = (df[['EmployeeID','FullName','Department','City','Salary','WeightedScore']]
                 .sort_values(['Department','WeightedScore'], ascending=[True, False])
                 .groupby('Department')
                 .head(3))

Cannot calculate WeightedScore. Missing columns: PerformanceScore, ProjectsCompleted. Load a dataset containing these columns.


KeyError: 'PerformanceScore'